# Niveau 3 — Inspection multizone

Inspection uniquement des candidats encore non examinés.


In [1]:

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from PIL import Image


ROOT = Path.cwd()

CSV_PATH = (
    ROOT
    / "data"
    / "niveau3_multizone"
    / "candidats"
    / "candidats_uniques.csv"
)

CROPS_DIR = (
    ROOT
    / "data"
    / "niveau3_multizone"
    / "candidats"
    / "revue"
    / "crops"
)

CONTEXTS_DIR = (
    ROOT
    / "data"
    / "niveau3_multizone"
    / "candidats"
    / "revue"
    / "contextes"
)


# ============================================================
# CHARGEMENT
# ============================================================

df = pd.read_csv(CSV_PATH)

# Correction robuste du bug Pandas float64
for col in [
    "review_status",
    "review_comment",
    "review_origin",
]:
    if col not in df.columns:
        df[col] = ""

    df[col] = (
        df[col]
        .fillna("")
        .astype("object")
    )


# Seulement les candidats encore non examinés
review_indices = df[
    df["review_status"]
    .astype(str)
    .str.strip()
    .eq("")
].index.tolist()


print("Candidats totaux :", len(df))
print(
    "Déjà examinés :",
    len(df) - len(review_indices)
)
print(
    "À examiner :",
    len(review_indices)
)


if len(review_indices) != 25:
    print(
        "⚠️ On attend normalement 25 candidats."
    )


# ============================================================
# ETAT
# ============================================================

current_position = 0


# ============================================================
# WIDGETS
# ============================================================

output = widgets.Output()

comment = widgets.Textarea(
    description="Commentaire :",
    placeholder="Optionnel...",
    layout=widgets.Layout(
        width="750px",
        height="80px",
    ),
)

btn_plausible = widgets.Button(
    description="✅ Plausible",
    button_style="success",
)

btn_false = widgets.Button(
    description="❌ Faux positif",
    button_style="danger",
)

btn_uncertain = widgets.Button(
    description="❓ Incertain",
    button_style="warning",
)

btn_prev = widgets.Button(
    description="⬅ Précédent",
)

btn_next = widgets.Button(
    description="Suivant ➡",
)


# ============================================================
# SAUVEGARDE
# ============================================================

def save():
    df.to_csv(
        CSV_PATH,
        index=False,
    )


# ============================================================
# AFFICHAGE
# ============================================================

def show_candidate():

    global current_position

    with output:

        clear_output(
            wait=True
        )

        if not review_indices:

            print(
                "✅ Aucun candidat à examiner."
            )
            return


        idx = review_indices[
            current_position
        ]

        row = df.loc[idx]

        uid = str(
            row["unique_candidate_id"]
        )

        crop_path = (
            CROPS_DIR
            / f"{uid}.jpg"
        )

        context_path = (
            CONTEXTS_DIR
            / f"{uid}.jpg"
        )


        print(
            f"Candidat "
            f"{current_position + 1}/"
            f"{len(review_indices)}"
        )

        print(
            f"ID : {uid}"
        )

        print(
            f"Zone : "
            f"{row['zone_id']}"
        )

        print(
            f"Confiance : "
            f"{float(row['confidence']):.3f}"
        )

        print(
            f"Distance ANFR : "
            f"{float(row['nearest_ANFR_distance_m']):.1f} m"
        )

        print(
            f"GPS : "
            f"{float(row['latitude']):.6f}, "
            f"{float(row['longitude']):.6f}"
        )

        print(
            f"Détections dans le cluster : "
            f"{int(row['detections_in_cluster'])}"
        )

        status = str(
            row["review_status"]
        ).strip()

        print(
            "Statut :",
            status
            if status
            else "non examiné"
        )


        comment.value = str(
            row["review_comment"]
        )


        fig, ax = plt.subplots(
            1,
            2,
            figsize=(13, 6),
        )


        if context_path.exists():

            context = Image.open(
                context_path
            )

            ax[0].imshow(
                context
            )

            ax[0].set_title(
                "Contexte"
            )

        else:

            ax[0].text(
                0.5,
                0.5,
                "Contexte absent",
                ha="center",
            )


        if crop_path.exists():

            crop = Image.open(
                crop_path
            )

            ax[1].imshow(
                crop
            )

            ax[1].set_title(
                "Crop détecté"
            )

        else:

            ax[1].text(
                0.5,
                0.5,
                "Crop absent",
                ha="center",
            )


        ax[0].axis("off")
        ax[1].axis("off")

        plt.tight_layout()
        plt.show()


# ============================================================
# ACTIONS
# ============================================================

def set_status(status):

    global current_position

    idx = review_indices[
        current_position
    ]


    df.at[
        idx,
        "review_status"
    ] = status

    df.at[
        idx,
        "review_comment"
    ] = comment.value

    df.at[
        idx,
        "review_origin"
    ] = "manual_multizone"


    save()


    if (
        current_position
        < len(review_indices) - 1
    ):

        current_position += 1

        show_candidate()

    else:

        with output:

            clear_output(
                wait=True
            )

            print(
                "✅ Les 25 candidats "
                "ont été parcourus."
            )

            remaining = int(
                df[
                    "review_status"
                ]
                .astype(str)
                .str.strip()
                .eq("")
                .sum()
            )

            print(
                "Candidats encore "
                "non examinés :",
                remaining
            )


def on_plausible(_):
    set_status(
        "plausible"
    )


def on_false(_):
    set_status(
        "faux_positif"
    )


def on_uncertain(_):
    set_status(
        "incertain"
    )


def on_prev(_):

    global current_position

    idx = review_indices[
        current_position
    ]

    df.at[
        idx,
        "review_comment"
    ] = comment.value

    save()


    if current_position > 0:
        current_position -= 1

    show_candidate()


def on_next(_):

    global current_position

    idx = review_indices[
        current_position
    ]

    df.at[
        idx,
        "review_comment"
    ] = comment.value

    save()


    if (
        current_position
        < len(review_indices) - 1
    ):
        current_position += 1

    show_candidate()


btn_plausible.on_click(
    on_plausible
)

btn_false.on_click(
    on_false
)

btn_uncertain.on_click(
    on_uncertain
)

btn_prev.on_click(
    on_prev
)

btn_next.on_click(
    on_next
)


buttons = widgets.HBox(
    [
        btn_plausible,
        btn_false,
        btn_uncertain,
        btn_prev,
        btn_next,
    ]
)


display(output)
display(comment)
display(buttons)

show_candidate()


Candidats totaux : 46
Déjà examinés : 21
À examiner : 25


Output()

Textarea(value='', description='Commentaire :', layout=Layout(height='80px', width='750px'), placeholder='Opti…